# Graph-ready longitudinal cohort

## Cohort

Patients with a recorded ischemic heart disease diagnosis:

- SNOMED CT code: `414545008`
- Description: `Ischemic heart disease (disorder)`
- Cohort size: 13 patients

## Index event

For each patient, the earliest recorded ischemic heart disease condition is
used as the index diagnosis date.

## Analysis window

The initial graph slice contains clinically relevant events occurring:

- 30 days before the index diagnosis
- 90 days after the index diagnosis

The window was selected as a deliberately narrow MVP scope. It provides enough
pre-diagnosis context and post-diagnosis follow-up to demonstrate longitudinal
GraphRAG without loading each patient's complete medical history.

This window is a configurable engineering decision, not a clinical standard.

## Condition interval behavior

Conditions starting within the window are included.

Conditions that started before the window are also included when their stop date
is null or overlaps the window. These represent potentially active historical
conditions and should not be interpreted as newly diagnosed during the window.

## Deferred scope

Observations and care plans remain available in Silver layers but are not loaded into
the first Neo4j graph slice.

In [0]:
from pyspark.sql import functions as F

CATALOG = "patient_kg_dev"
SILVER = f"{CATALOG}.silver"
QUALITY = f"{CATALOG}.quality"
GRAPH = f"{CATALOG}.graph_ready"

IHD_CODE = "414545008"

DAYS_BEFORE = 30
DAYS_AFTER = 90

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GRAPH}")

print("Silver:", SILVER)
print("Quality:", QUALITY)
print("Graph-ready:", GRAPH)

In [0]:
quality_history = spark.table(f"{QUALITY}.check_results")

latest_quality_run = (
    quality_history
    .orderBy(F.desc("quality_run_at"))
    .select("quality_run_id")
    .first()["quality_run_id"]
)

latest_quality_results = (
    quality_history
    .filter(F.col("quality_run_id") == latest_quality_run)
)

blocking_quality_failures = (
    latest_quality_results
    .filter(
        (F.col("severity") == "ERROR") &
        (F.col("status") == "FAIL")
    )
)

display(blocking_quality_failures)

assert blocking_quality_failures.count() == 0, (
    "The latest quality run has blocking failures."
)

print("QUALITY GATE PASSED")
print("Quality run:", latest_quality_run)

In [0]:
cohort_index = (
    spark.table(f"{SILVER}.conditions")
    .filter(F.col("code") == IHD_CODE)
    .groupBy("patient_id")
    .agg(
        F.min("start_date").alias("index_diagnosis_date")
    )
    .withColumn(
        "window_start",
        F.date_sub(
            F.col("index_diagnosis_date"),
            DAYS_BEFORE
        )
    )
    .withColumn(
        "window_end",
        F.date_add(
            F.col("index_diagnosis_date"),
            DAYS_AFTER
        )
    )
    .withColumn("cohort_id", F.lit("ischemic_heart_disease_v1"))
    .withColumn("days_before", F.lit(DAYS_BEFORE))
    .withColumn("days_after", F.lit(DAYS_AFTER))
)

assert cohort_index.count() == 13, (
    "Expected 13 patients in the IHD cohort."
)

assert (
    cohort_index
    .filter(F.col("index_diagnosis_date").isNull())
    .count()
    == 0
), "A cohort patient is missing an index diagnosis date."

cohort_index.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GRAPH}.cohort_index")

display(cohort_index.orderBy("index_diagnosis_date"))

# Patient Node

In [0]:
patient_nodes = (
    spark.table(f"{SILVER}.patients").alias("p")
    .join(
        cohort_index.alias("c"),
        F.col("p.patient_id") == F.col("c.patient_id"),
        "inner"
    )
    .select(
        F.concat(
            F.lit("patient:"),
            F.col("p.patient_id")
        ).alias("node_id"),

        F.lit("Patient").alias("node_label"),

        F.col("p.patient_id"),
        F.col("p.birth_date"),
        F.col("p.death_date"),
        F.col("p.gender"),
        F.col("p.race"),
        F.col("p.ethnicity"),
        F.col("p.city"),
        F.col("p.state"),
        F.col("p.healthcare_expenses"),
        F.col("p.healthcare_coverage"),
        F.col("p.income"),

        F.col("c.index_diagnosis_date"),
        F.col("c.window_start"),
        F.col("c.window_end"),
        F.col("c.cohort_id"),

        F.col("p._source_file"),
        F.col("p._row_content_sha256"),
        F.col("p._ingestion_run_id"),
        F.current_timestamp().alias("_graph_created_at")
    )
)

patient_nodes.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GRAPH}.patient_nodes")

print("Patient nodes:", patient_nodes.count())

# Encounter node

In [0]:
encounter_nodes = (
    spark.table(f"{SILVER}.encounters").alias("e")
    .join(
        cohort_index.alias("c"),
        F.col("e.patient_id") == F.col("c.patient_id"),
        "inner"
    )
    .filter(
        (
            F.to_date(F.col("e.start_at")) <=
            F.col("c.window_end")
        ) &
        (
            F.to_date(
                F.coalesce(
                    F.col("e.stop_at"),
                    F.col("e.start_at")
                )
            ) >= F.col("c.window_start")
        )
    )
    .select(
        F.concat(
            F.lit("encounter:"),
            F.col("e.encounter_id")
        ).alias("node_id"),

        F.lit("Encounter").alias("node_label"),

        F.col("e.encounter_id"),
        F.col("e.patient_id"),
        F.col("e.start_at"),
        F.col("e.stop_at"),
        F.col("e.encounter_class"),
        F.col("e.code"),
        F.col("e.description_source"),
        F.col("e.organization_id"),
        F.col("e.provider_id"),
        F.col("e.payer_id"),
        F.col("e.reason_code"),
        F.col("e.reason_description_source"),
        F.col("e.base_cost"),
        F.col("e.total_claim_cost"),
        F.col("e.payer_coverage"),

        F.col("c.index_diagnosis_date"),
        F.col("c.window_start"),
        F.col("c.window_end"),

        F.col("e._source_file"),
        F.col("e._row_content_sha256"),
        F.col("e._ingestion_run_id"),
        F.current_timestamp().alias("_graph_created_at")
    )
    .dropDuplicates(["node_id"])
)

encounter_nodes.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GRAPH}.encounter_nodes")

print("Encounter nodes:", encounter_nodes.count())

# Condition-event node

In [0]:
condition_nodes = (
    spark.table(f"{SILVER}.conditions")
    .dropDuplicates(["_row_content_sha256"])
    .alias("condition")
    .join(
        cohort_index.alias("c"),
        F.col("condition.patient_id") == F.col("c.patient_id"),
        "inner"
    )
    .filter(
        (
            F.col("condition.start_date") <=
            F.col("c.window_end")
        ) &
        (
            F.coalesce(
                F.col("condition.stop_date"),
                F.col("c.window_end")
            ) >= F.col("c.window_start")
        )
    )
    .select(
        F.concat(
            F.lit("condition:"),
            F.col("condition._row_content_sha256")
        ).alias("node_id"),

        F.lit("ConditionEvent").alias("node_label"),

        F.col("condition.condition_key_sha256"),
        F.col("condition.patient_id"),
        F.col("condition.encounter_id"),
        F.col("condition.start_date"),
        F.col("condition.stop_date"),
        F.col("condition.code_system"),
        F.col("condition.code"),
        F.col("condition.description_source"),

        (
            F.col("condition.code") == IHD_CODE
        ).alias("is_index_condition_type"),

        (
            F.col("condition.start_date") ==
            F.col("c.index_diagnosis_date")
        ).alias("occurs_on_index_date"),

        F.col("c.index_diagnosis_date"),

        F.col("condition._source_file"),
        F.col("condition._row_content_sha256"),
        F.col("condition._ingestion_run_id"),
        F.current_timestamp().alias("_graph_created_at")
    )
    .dropDuplicates(["node_id"])
)

condition_nodes.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GRAPH}.condition_nodes")

print("Condition nodes:", condition_nodes.count())

# Medication-Event Node

In [0]:
medication_nodes = (
    spark.table(f"{SILVER}.medications")
    .dropDuplicates(["_row_content_sha256"])
    .alias("medication")
    .join(
        cohort_index.alias("c"),
        F.col("medication.patient_id") == F.col("c.patient_id"),
        "inner"
    )
    .filter(
        (
            F.to_date(F.col("medication.start_at")) <=
            F.col("c.window_end")
        ) &
        (
            F.to_date(
                F.coalesce(
                    F.col("medication.stop_at"),
                    F.to_timestamp(F.col("c.window_end"))
                )
            ) >= F.col("c.window_start")
        )
    )
    .select(
        F.concat(
            F.lit("medication:"),
            F.col("medication._row_content_sha256")
        ).alias("node_id"),

        F.lit("MedicationEvent").alias("node_label"),

        F.col("medication.medication_key_sha256"),
        F.col("medication.patient_id"),
        F.col("medication.encounter_id"),
        F.col("medication.payer_id"),
        F.col("medication.start_at"),
        F.col("medication.stop_at"),
        F.col("medication.code"),
        F.col("medication.description_source"),
        F.col("medication.base_cost"),
        F.col("medication.payer_coverage"),
        F.col("medication.dispenses"),
        F.col("medication.total_cost"),
        F.col("medication.reason_code"),
        F.col("medication.reason_description_source"),

        F.col("c.index_diagnosis_date"),

        F.col("medication._source_file"),
        F.col("medication._row_content_sha256"),
        F.col("medication._ingestion_run_id"),
        F.current_timestamp().alias("_graph_created_at")
    )
    .dropDuplicates(["node_id"])
)

medication_nodes.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GRAPH}.medication_nodes")

print("Medication nodes:", medication_nodes.count())

# Procedure-event node

In [0]:
procedure_nodes = (
    spark.table(f"{SILVER}.procedures")
    .dropDuplicates(["_row_content_sha256"])
    .alias("procedure")
    .join(
        cohort_index.alias("c"),
        F.col("procedure.patient_id") == F.col("c.patient_id"),
        "inner"
    )
    .filter(
        (
            F.to_date(F.col("procedure.start_at")) <=
            F.col("c.window_end")
        ) &
        (
            F.to_date(
                F.coalesce(
                    F.col("procedure.stop_at"),
                    F.col("procedure.start_at")
                )
            ) >= F.col("c.window_start")
        )
    )
    .select(
        F.concat(
            F.lit("procedure:"),
            F.col("procedure._row_content_sha256")
        ).alias("node_id"),

        F.lit("ProcedureEvent").alias("node_label"),

        F.col("procedure.procedure_key_sha256"),
        F.col("procedure.patient_id"),
        F.col("procedure.encounter_id"),
        F.col("procedure.start_at"),
        F.col("procedure.stop_at"),
        F.col("procedure.code_system"),
        F.col("procedure.code"),
        F.col("procedure.description_source"),
        F.col("procedure.base_cost"),
        F.col("procedure.reason_code"),
        F.col("procedure.reason_description_source"),

        F.col("c.index_diagnosis_date"),

        F.col("procedure._source_file"),
        F.col("procedure._row_content_sha256"),
        F.col("procedure._ingestion_run_id"),
        F.current_timestamp().alias("_graph_created_at")
    )
    .dropDuplicates(["node_id"])
)

procedure_nodes.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GRAPH}.procedure_nodes")

print("Procedure nodes:", procedure_nodes.count())

# Relationship

In [0]:
def build_relationship(
    dataframe,
    source_column,
    target_column,
    relationship_type,
    patient_column="patient_id",
    encounter_column=None,
    event_date_column=None
):
    encounter_expression = (
        F.col(encounter_column)
        if encounter_column
        else F.lit(None).cast("string")
    )

    event_date_expression = (
        F.col(event_date_column).cast("timestamp")
        if event_date_column
        else F.lit(None).cast("timestamp")
    )

    return (
        dataframe
        .select(
            F.sha2(
                F.concat_ws(
                    "||",
                    F.col(source_column),
                    F.lit(relationship_type),
                    F.col(target_column)
                ),
                256
            ).alias("relationship_id"),

            F.col(source_column).alias("source_node_id"),
            F.col(target_column).alias("target_node_id"),
            F.lit(relationship_type).alias("relationship_type"),

            F.col(patient_column).alias("patient_id"),
            encounter_expression.alias("encounter_id"),
            event_date_expression.alias("event_at"),

            F.current_timestamp().alias("_graph_created_at")
        )
        .dropDuplicates(["relationship_id"])
    )

# Patient-to-encounter relationship

In [0]:
patient_encounter_input = (
    encounter_nodes
    .select(
        F.concat(
            F.lit("patient:"),
            F.col("patient_id")
        ).alias("patient_node_id"),

        F.col("node_id").alias("encounter_node_id"),
        "patient_id",
        "encounter_id",
        "start_at"
    )
)

patient_encounter_relationships = build_relationship(
    dataframe=patient_encounter_input,
    source_column="patient_node_id",
    target_column="encounter_node_id",
    relationship_type="HAS_ENCOUNTER",
    patient_column="patient_id",
    encounter_column="encounter_id",
    event_date_column="start_at"
)

# Patient-to-condition relationship

In [0]:
patient_condition_input = (
    condition_nodes
    .select(
        F.concat(
            F.lit("patient:"),
            F.col("patient_id")
        ).alias("patient_node_id"),

        F.col("node_id").alias("condition_node_id"),
        "patient_id",
        "encounter_id",
        "start_date"
    )
)

patient_condition_relationships = build_relationship(
    dataframe=patient_condition_input,
    source_column="patient_node_id",
    target_column="condition_node_id",
    relationship_type="HAS_CONDITION",
    patient_column="patient_id",
    encounter_column="encounter_id",
    event_date_column="start_date"
)

# Patient-to-medication relationship

In [0]:
patient_medication_input = (
    medication_nodes
    .select(
        F.concat(
            F.lit("patient:"),
            F.col("patient_id")
        ).alias("patient_node_id"),

        F.col("node_id").alias("medication_node_id"),
        "patient_id",
        "encounter_id",
        "start_at"
    )
)

patient_medication_relationships = build_relationship(
    dataframe=patient_medication_input,
    source_column="patient_node_id",
    target_column="medication_node_id",
    relationship_type="HAS_MEDICATION",
    patient_column="patient_id",
    encounter_column="encounter_id",
    event_date_column="start_at"
)

# Patient-to-procedure relationship

In [0]:
patient_procedure_input = (
    procedure_nodes
    .select(
        F.concat(
            F.lit("patient:"),
            F.col("patient_id")
        ).alias("patient_node_id"),

        F.col("node_id").alias("procedure_node_id"),
        "patient_id",
        "encounter_id",
        "start_at"
    )
)

patient_procedure_relationships = build_relationship(
    dataframe=patient_procedure_input,
    source_column="patient_node_id",
    target_column="procedure_node_id",
    relationship_type="HAS_PROCEDURE",
    patient_column="patient_id",
    encounter_column="encounter_id",
    event_date_column="start_at"
)

# Encounter-to-condition relationship

In [0]:
encounter_condition_input = (
    condition_nodes.alias("condition")
    .filter(F.col("condition.encounter_id").isNotNull())
    .join(
        encounter_nodes.alias("encounter"),
        (
            F.col("condition.encounter_id") ==
            F.col("encounter.encounter_id")
        ) &
        (
            F.col("condition.patient_id") ==
            F.col("encounter.patient_id")
        ),
        "inner"
    )
    .select(
        F.col("encounter.node_id").alias("encounter_node_id"),
        F.col("condition.node_id").alias("condition_node_id"),
        F.col("condition.patient_id").alias("patient_id"),
        F.col("condition.encounter_id").alias("encounter_id"),
        F.col("condition.start_date").alias("event_date")
    )
)

encounter_condition_relationships = build_relationship(
    dataframe=encounter_condition_input,
    source_column="encounter_node_id",
    target_column="condition_node_id",
    relationship_type="HAS_CONDITION",
    patient_column="patient_id",
    encounter_column="encounter_id",
    event_date_column="event_date"
)

# Encounter-to-medication relationship

In [0]:
encounter_medication_input = (
    medication_nodes.alias("medication")
    .filter(F.col("medication.encounter_id").isNotNull())
    .join(
        encounter_nodes.alias("encounter"),
        (
            F.col("medication.encounter_id") ==
            F.col("encounter.encounter_id")
        ) &
        (
            F.col("medication.patient_id") ==
            F.col("encounter.patient_id")
        ),
        "inner"
    )
    .select(
        F.col("encounter.node_id").alias("encounter_node_id"),
        F.col("medication.node_id").alias("medication_node_id"),
        F.col("medication.patient_id").alias("patient_id"),
        F.col("medication.encounter_id").alias("encounter_id"),
        F.col("medication.start_at").alias("event_at")
    )
)

encounter_medication_relationships = build_relationship(
    dataframe=encounter_medication_input,
    source_column="encounter_node_id",
    target_column="medication_node_id",
    relationship_type="HAS_MEDICATION",
    patient_column="patient_id",
    encounter_column="encounter_id",
    event_date_column="event_at"
)

# Encounter-to-procedure relationship

In [0]:
encounter_procedure_input = (
    procedure_nodes.alias("procedure")
    .filter(F.col("procedure.encounter_id").isNotNull())
    .join(
        encounter_nodes.alias("encounter"),
        (
            F.col("procedure.encounter_id") ==
            F.col("encounter.encounter_id")
        ) &
        (
            F.col("procedure.patient_id") ==
            F.col("encounter.patient_id")
        ),
        "inner"
    )
    .select(
        F.col("encounter.node_id").alias("encounter_node_id"),
        F.col("procedure.node_id").alias("procedure_node_id"),
        F.col("procedure.patient_id").alias("patient_id"),
        F.col("procedure.encounter_id").alias("encounter_id"),
        F.col("procedure.start_at").alias("event_at")
    )
)

encounter_procedure_relationships = build_relationship(
    dataframe=encounter_procedure_input,
    source_column="encounter_node_id",
    target_column="procedure_node_id",
    relationship_type="HAS_PROCEDURE",
    patient_column="patient_id",
    encounter_column="encounter_id",
    event_date_column="event_at"
)

# Combine and save relationship

In [0]:
relationships = (
    patient_encounter_relationships
    .unionByName(patient_condition_relationships)
    .unionByName(patient_medication_relationships)
    .unionByName(patient_procedure_relationships)
    .unionByName(encounter_condition_relationships)
    .unionByName(encounter_medication_relationships)
    .unionByName(encounter_procedure_relationships)
    .dropDuplicates(["relationship_id"])
)

relationships.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GRAPH}.relationships")

display(
    relationships
    .groupBy("relationship_type")
    .count()
    .orderBy("relationship_type")
)

# Node index

In [0]:
node_index = (
    patient_nodes.select(
        "node_id",
        "node_label",
        "patient_id"
    )
    .unionByName(
        encounter_nodes.select(
            "node_id",
            "node_label",
            "patient_id"
        )
    )
    .unionByName(
        condition_nodes.select(
            "node_id",
            "node_label",
            "patient_id"
        )
    )
    .unionByName(
        medication_nodes.select(
            "node_id",
            "node_label",
            "patient_id"
        )
    )
    .unionByName(
        procedure_nodes.select(
            "node_id",
            "node_label",
            "patient_id"
        )
    )
)

node_index.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GRAPH}.node_index")

# Validate node uniqueness

In [0]:
duplicate_node_ids = (
    node_index
    .groupBy("node_id")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_node_ids)

assert duplicate_node_ids.count() == 0, (
    "Graph contains duplicate node IDs."
)

# Validate relationship uniqueness

In [0]:
duplicate_relationship_ids = (
    relationships
    .groupBy("relationship_id")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_relationship_ids)

assert duplicate_relationship_ids.count() == 0, (
    "Graph contains duplicate relationship IDs."
)

# Validate dangling relationships

In [0]:
valid_node_ids = node_index.select("node_id").distinct()

dangling_sources = (
    relationships
    .select(
        F.col("source_node_id").alias("node_id")
    )
    .join(valid_node_ids, "node_id", "left_anti")
)

dangling_targets = (
    relationships
    .select(
        F.col("target_node_id").alias("node_id")
    )
    .join(valid_node_ids, "node_id", "left_anti")
)

display(dangling_sources)
display(dangling_targets)

assert dangling_sources.count() == 0, (
    "Graph contains relationships with missing source nodes."
)

assert dangling_targets.count() == 0, (
    "Graph contains relationships with missing target nodes."
)

# Confirm all patients have their IHD diagnosis

In [0]:
patients_with_ihd = (
    condition_nodes
    .filter(F.col("code") == IHD_CODE)
    .select("patient_id")
    .distinct()
    .count()
)

print("Patients represented:", patient_nodes.count())
print("Patients with IHD condition:", patients_with_ihd)

assert patient_nodes.count() == 13
assert patients_with_ihd == 13

# Graph Reconcillation Summary

In [0]:
graph_summary = [
    ("Patient", patient_nodes.count()),
    ("Encounter", encounter_nodes.count()),
    ("ConditionEvent", condition_nodes.count()),
    ("MedicationEvent", medication_nodes.count()),
    ("ProcedureEvent", procedure_nodes.count()),
    ("TOTAL_NODES", node_index.count()),
    ("TOTAL_RELATIONSHIPS", relationships.count())
]

graph_summary_df = spark.createDataFrame(
    graph_summary,
    ["graph_object", "row_count"]
)

display(graph_summary_df)

In [0]:
sample_patient_id = (
    cohort_index
    .orderBy("index_diagnosis_date")
    .select("patient_id")
    .first()["patient_id"]
)

print("Sample patient:", sample_patient_id)

sample_timeline = (
    condition_nodes
    .filter(F.col("patient_id") == sample_patient_id)
    .select(
        "patient_id",
        F.col("start_date").cast("timestamp").alias("event_at"),
        F.lit("Condition").alias("event_type"),
        "code",
        "description_source"
    )
    .unionByName(
        medication_nodes
        .filter(F.col("patient_id") == sample_patient_id)
        .select(
            "patient_id",
            F.col("start_at").alias("event_at"),
            F.lit("Medication").alias("event_type"),
            "code",
            "description_source"
        )
    )
    .unionByName(
        procedure_nodes
        .filter(F.col("patient_id") == sample_patient_id)
        .select(
            "patient_id",
            F.col("start_at").alias("event_at"),
            F.lit("Procedure").alias("event_type"),
            "code",
            "description_source"
        )
    )
    .orderBy("event_at")
)

display(sample_timeline)

In [0]:
required_graph_tables = [
    "cohort_index",
    "patient_nodes",
    "encounter_nodes",
    "condition_nodes",
    "medication_nodes",
    "procedure_nodes",
    "node_index",
    "relationships"
]

for table_name in required_graph_tables:
    full_table_name = f"{GRAPH}.{table_name}"

    assert spark.catalog.tableExists(full_table_name), (
        f"Missing Graph-ready table: {full_table_name}"
    )

    print(
        full_table_name,
        spark.table(full_table_name).count()
    )

print("GRAPH-READY LAYER COMPLETED SUCCESSFULLY")
print("The next step is loading these nodes and relationships into Neo4j.")

In [0]:
graph_run_metadata = spark.createDataFrame(
    [
        (
            "ischemic_heart_disease_v1",
            IHD_CODE,
            "Ischemic heart disease (disorder)",
            "earliest_recorded_ihd_condition",
            DAYS_BEFORE,
            DAYS_AFTER,
            "Configurable MVP scope; not a clinical standard",
            latest_quality_run
        )
    ],
    [
        "cohort_id",
        "index_condition_code",
        "index_condition_description",
        "index_event_rule",
        "days_before",
        "days_after",
        "window_rationale",
        "source_quality_run_id"
    ]
).withColumn(
    "graph_created_at",
    F.current_timestamp()
)

graph_run_metadata.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GRAPH}.graph_run_metadata")

display(graph_run_metadata)